In [ ]:
print("Retail sales notebook is ready")

In [ ]:
from pathlib import Path

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_folder = current_folder.parent
else:
    project_folder = current_folder

data_folder = project_folder / "01_Raw_Data"

print("CSV files found:")

for file in data_folder.glob("*.csv"):
    print("-", file.name)

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import statsmodels

print("All analysis packages are ready!")

In [ ]:
sales = pd.read_csv(data_folder / "sales data-set.csv")
features = pd.read_csv(data_folder / "Features data set.csv")
stores = pd.read_csv(data_folder / "stores data-set.csv")

print("Sales table:", sales.shape)
print("Features table:", features.shape)
print("Stores table:", stores.shape)

In [ ]:
print("SALES DATA")
display(sales.head())

print("FEATURES DATA")
display(features.head())

print("STORES DATA")
display(stores.head())

## 1. Overview of the Datasets

This project uses three datasets: Sales, Features and Stores. The following checks examine their size, columns, missing values and duplicate records.

In [ ]:
dataset_overview = pd.DataFrame({
    "Dataset": ["Sales", "Features", "Stores"],
    "Rows": [
        len(sales),
        len(features),
        len(stores)
    ],
    "Columns": [
        sales.shape[1],
        features.shape[1],
        stores.shape[1]
    ],
    "Missing Cells": [
        sales.isna().sum().sum(),
        features.isna().sum().sum(),
        stores.isna().sum().sum()
    ],
    "Duplicate Rows": [
        sales.duplicated().sum(),
        features.duplicated().sum(),
        stores.duplicated().sum()
    ]
})

display(dataset_overview)

In [ ]:
print("SALES COLUMNS")
print(sales.columns.tolist())

print("\nFEATURES COLUMNS")
print(features.columns.tolist())

print("\nSTORES COLUMNS")
print(stores.columns.tolist())

### 1.1 Column Structure and Missing Values

The following table checks the data type, missing values and number of unique values in every column.

In [ ]:
def create_column_summary(dataframe, dataset_name):
    return pd.DataFrame({
        "Dataset": [dataset_name] * len(dataframe.columns),
        "Column": dataframe.columns,
        "Data Type": dataframe.dtypes.astype(str).values,
        "Missing Values": dataframe.isna().sum().values,
        "Unique Values": dataframe.nunique(dropna=True).values
    })

sales_columns = create_column_summary(sales, "Sales")
features_columns = create_column_summary(features, "Features")
stores_columns = create_column_summary(stores, "Stores")

column_summary = pd.concat(
    [sales_columns, features_columns, stores_columns],
    ignore_index=True
)

display(column_summary)

In [ ]:
missing_columns = column_summary[
    column_summary["Missing Values"] > 0
].sort_values(
    by="Missing Values",
    ascending=False
)

display(missing_columns)

### 1.2 Variables and Their Importance

| Dataset | Variable | Meaning and importance |
|---|---|---|
| Sales | Store | Identifies the store where the sale happened. |
| Sales | Dept | Identifies the department inside the store. |
| Sales | Date | Shows the week when the sale was recorded. |
| Sales | Weekly_Sales | Shows weekly sales for a department. This is the main target variable that will be predicted. |
| Sales | IsHoliday | Shows whether the week was a special holiday week. |
| Features | Store | Connects the Features data with a store. |
| Features | Date | Connects the Features data with the correct sales week. |
| Features | Temperature | Shows the average regional temperature for the week. |
| Features | Fuel_Price | Shows the regional fuel price, which may affect customer spending. |
| Features | MarkDown1–5 | Shows different promotional markdown activities that may influence sales. |
| Features | CPI | Shows changes in consumer prices and the economy. |
| Features | Unemployment | Shows the regional unemployment rate. |
| Features | IsHoliday | Identifies special holiday weeks. |
| Stores | Store | Provides the identification number of each store. |
| Stores | Type | Divides stores into types A, B and C. |
| Stores | Size | Shows the physical size of each store. |

## 2. Data Cleaning and Preparation

Copies of the original datasets were created before cleaning. This protected the raw files and allowed the cleaning steps to be checked and repeated.

In [ ]:
sales_clean = sales.copy()
features_clean = features.copy()
stores_clean = stores.copy()

sales_clean.columns = sales_clean.columns.str.strip()
features_clean.columns = features_clean.columns.str.strip()
stores_clean.columns = stores_clean.columns.str.strip()

print("Clean copies created successfully.")

In [ ]:
sales_clean["Date"] = pd.to_datetime(
    sales_clean["Date"],
    format="%d/%m/%Y",
    errors="coerce"
)

features_clean["Date"] = pd.to_datetime(
    features_clean["Date"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("Invalid Sales dates:", sales_clean["Date"].isna().sum())
print("Invalid Features dates:", features_clean["Date"].isna().sum())

print(
    "Sales date range:",
    sales_clean["Date"].min().date(),
    "to",
    sales_clean["Date"].max().date()
)

print(
    "Features date range:",
    features_clean["Date"].min().date(),
    "to",
    features_clean["Date"].max().date()
)

### 2.1 Data-Type Correction

Store and department identifiers were converted to whole numbers. Sales, economic and markdown variables were converted to numerical values. Holiday fields were converted to Boolean values, while store types were cleaned as text.

In [ ]:
def clean_boolean_column(column):
    if pd.api.types.is_bool_dtype(column):
        return column.astype("boolean")
    
    return (
        column.astype("string")
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
        .astype("boolean")
    )

sales_clean["IsHoliday"] = clean_boolean_column(
    sales_clean["IsHoliday"]
)

features_clean["IsHoliday"] = clean_boolean_column(
    features_clean["IsHoliday"]
)

print("Holiday columns cleaned.")

In [ ]:
sales_clean["Store"] = pd.to_numeric(
    sales_clean["Store"],
    errors="coerce"
).astype("Int64")

sales_clean["Dept"] = pd.to_numeric(
    sales_clean["Dept"],
    errors="coerce"
).astype("Int64")

sales_clean["Weekly_Sales"] = pd.to_numeric(
    sales_clean["Weekly_Sales"],
    errors="coerce"
)

print(sales_clean.dtypes)

In [ ]:
features_clean["Store"] = pd.to_numeric(
    features_clean["Store"],
    errors="coerce"
).astype("Int64")

feature_number_columns = [
    "Temperature",
    "Fuel_Price",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5",
    "CPI",
    "Unemployment"
]

for column in feature_number_columns:
    features_clean[column] = pd.to_numeric(
        features_clean[column],
        errors="coerce"
    )

print(features_clean.dtypes)

In [ ]:
stores_clean["Store"] = pd.to_numeric(
    stores_clean["Store"],
    errors="coerce"
).astype("Int64")

stores_clean["Type"] = (
    stores_clean["Type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

stores_clean["Size"] = pd.to_numeric(
    stores_clean["Size"],
    errors="coerce"
).astype("Int64")

print(stores_clean.dtypes)

In [ ]:
print("Missing essential Sales values:")
print(
    sales_clean[
        ["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]
    ].isna().sum()
)

print("\nStore types:")
print(stores_clean["Type"].value_counts())

### 2.2 Duplicate and Unusual Sales Checks

Duplicate records were checked using the main identifying columns in each dataset. Negative sales were also examined. Negative values were flagged but not deleted because they may represent customer returns, refunds or accounting corrections.

In [ ]:
duplicate_check = pd.DataFrame({
    "Dataset": ["Sales", "Features", "Stores"],
    
    "Complete Duplicate Rows": [
        sales_clean.duplicated().sum(),
        features_clean.duplicated().sum(),
        stores_clean.duplicated().sum()
    ],
    
    "Duplicate Key Records": [
        sales_clean.duplicated(
            subset=["Store", "Dept", "Date"]
        ).sum(),
        
        features_clean.duplicated(
            subset=["Store", "Date"]
        ).sum(),
        
        stores_clean.duplicated(
            subset=["Store"]
        ).sum()
    ]
})

display(duplicate_check)

In [ ]:
negative_sales_count = (
    sales_clean["Weekly_Sales"] < 0
).sum()

zero_sales_count = (
    sales_clean["Weekly_Sales"] == 0
).sum()

negative_sales_percentage = (
    negative_sales_count / len(sales_clean)
) * 100

print("Negative sales records:", negative_sales_count)
print("Zero sales records:", zero_sales_count)
print(
    "Negative sales percentage:",
    round(negative_sales_percentage, 3),
    "%"
)

In [ ]:
sales_clean["Sales_Status"] = np.select(
    [
        sales_clean["Weekly_Sales"] < 0,
        sales_clean["Weekly_Sales"] == 0
    ],
    [
        "Negative",
        "Zero"
    ],
    default="Positive"
)

print(sales_clean["Sales_Status"].value_counts())

In [ ]:
display(
    sales_clean[
        sales_clean["Sales_Status"] == "Negative"
    ].head(10)
)

### 2.3 Missing-Value Analysis

Missing values were mainly found in the markdown columns. These values were not immediately deleted or replaced because markdown information was only collected after November 2011 and was not available for every store and week. Keeping the missing values prevents the analysis from treating unknown information as confirmed zero markdown activity.

In [ ]:
features_missing = pd.DataFrame({
    "Column": features_clean.columns,
    "Missing Values": features_clean.isna().sum().values,
    "Missing Percentage": (
        features_clean.isna().mean().values * 100
    )
})

features_missing["Missing Percentage"] = (
    features_missing["Missing Percentage"].round(2)
)

features_missing = features_missing[
    features_missing["Missing Values"] > 0
].sort_values(
    by="Missing Values",
    ascending=False
)

display(features_missing)

In [ ]:
for column in ["CPI", "Unemployment"]:
    missing_dates = features_clean.loc[
        features_clean[column].isna(),
        "Date"
    ]
    
    print(f"{column} missing values:")
    print("First missing date:", missing_dates.min())
    print("Last missing date:", missing_dates.max())
    print()

In [ ]:
print(
    "Missing cells in Sales:",
    sales_clean.isna().sum().sum()
)

print(
    "Missing cells in Stores:",
    stores_clean.isna().sum().sum()
)

### 2.4 Data Integration

The Sales and Features datasets were joined using Store and Date because each feature record belongs to a particular store and week. The Stores dataset was then joined using Store. Left joins were used to keep every historical sales record. The merge results were checked to confirm that no sales records were lost.

In [ ]:
sales_features = sales_clean.merge(
    features_clean,
    on=["Store", "Date"],
    how="left",
    suffixes=("_Sales", "_Features"),
    validate="many_to_one",
    indicator=True
)

print("Original Sales rows:", len(sales_clean))
print("Rows after first merge:", len(sales_features))

print("\nMerge result:")
print(sales_features["_merge"].value_counts())

In [ ]:
holiday_comparison = sales_features[
    ["IsHoliday_Sales", "IsHoliday_Features"]
].dropna()

holiday_mismatches = (
    holiday_comparison["IsHoliday_Sales"]
    != holiday_comparison["IsHoliday_Features"]
).sum()

print("Holiday mismatches:", holiday_mismatches)

In [ ]:
analysis_data = sales_features.drop(
    columns=["IsHoliday_Features", "_merge"]
)

analysis_data = analysis_data.rename(
    columns={"IsHoliday_Sales": "IsHoliday"}
)

print("Duplicate holiday column removed.")

In [ ]:
analysis_data = analysis_data.merge(
    stores_clean,
    on="Store",
    how="left",
    validate="many_to_one",
    indicator="Store_Merge"
)

print(analysis_data["Store_Merge"].value_counts())

In [ ]:
analysis_data = analysis_data.drop(
    columns=["Store_Merge"]
)

print("Final table shape:", analysis_data.shape)
print("Rows preserved:", len(analysis_data) == len(sales_clean))

display(analysis_data.head())

### 2.5 Creation of Analysis Variables

Additional time, holiday and markdown variables were created to support trend analysis and predictive modelling. These included year, month, week, quarter, holiday name and total reported markdown.

In [ ]:
analysis_data["Year"] = analysis_data["Date"].dt.year
analysis_data["Month_Number"] = analysis_data["Date"].dt.month
analysis_data["Month_Name"] = analysis_data["Date"].dt.month_name()
analysis_data["Year_Month"] = (
    analysis_data["Date"].dt.to_period("M").astype(str)
)
analysis_data["Week_Number"] = (
    analysis_data["Date"].dt.isocalendar().week.astype("Int64")
)
analysis_data["Quarter"] = analysis_data["Date"].dt.quarter

print("Time columns created.")

In [ ]:
holiday_dates = (
    analysis_data.loc[
        analysis_data["IsHoliday"] == True,
        ["Date"]
    ]
    .drop_duplicates()
    .sort_values("Date")
)

display(holiday_dates)

In [ ]:
holiday_map = {
    pd.Timestamp("2010-02-12"): "Super Bowl",
    pd.Timestamp("2010-09-10"): "Labour Day",
    pd.Timestamp("2010-11-26"): "Thanksgiving",
    pd.Timestamp("2010-12-31"): "Christmas",
    
    pd.Timestamp("2011-02-11"): "Super Bowl",
    pd.Timestamp("2011-09-09"): "Labour Day",
    pd.Timestamp("2011-11-25"): "Thanksgiving",
    pd.Timestamp("2011-12-30"): "Christmas",
    
    pd.Timestamp("2012-02-10"): "Super Bowl",
    pd.Timestamp("2012-09-07"): "Labour Day"
}

analysis_data["Holiday_Name"] = (
    analysis_data["Date"]
    .map(holiday_map)
    .fillna("Non-holiday")
)

holiday_summary = (
    analysis_data.loc[
        analysis_data["IsHoliday"] == True,
        ["Date", "Holiday_Name"]
    ]
    .drop_duplicates()
    .sort_values("Date")
)

display(holiday_summary)

In [ ]:
unlabelled_holidays = analysis_data[
    (analysis_data["IsHoliday"] == True) &
    (analysis_data["Holiday_Name"] == "Non-holiday")
]

print(
    "Unlabelled holiday records:",
    len(unlabelled_holidays)
)

In [ ]:
markdown_columns = [
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

analysis_data["Total_MarkDown"] = (
    analysis_data[markdown_columns]
    .sum(axis=1, min_count=1)
)

analysis_data["Markdown_Reported"] = (
    analysis_data[markdown_columns]
    .notna()
    .any(axis=1)
)

print(
    analysis_data["Markdown_Reported"].value_counts()
)

### 2.6 Saving the Prepared Dataset

The final merged and prepared dataset was saved separately from the original files. This protected the raw data and created a reusable file for EDA, visualisation and predictive modelling.

In [ ]:
cleaned_folder = project_folder / "02_Cleaned_Data"
charts_folder = project_folder / "04_Charts"
screenshots_folder = project_folder / "05_Screenshots"
report_folder = project_folder / "06_Report"
ai_evidence_folder = project_folder / "07_AI_Evidence"
demonstration_folder = project_folder / "08_Demonstration"

folders = [
    cleaned_folder,
    charts_folder,
    screenshots_folder,
    report_folder,
    ai_evidence_folder,
    demonstration_folder
]

for folder in folders:
    folder.mkdir(exist_ok=True)

print("All project folders are ready.")

In [ ]:
cleaned_file = (
    cleaned_folder / "retail_analysis_data.csv"
)

analysis_data.to_csv(
    cleaned_file,
    index=False
)

dataset_overview.to_csv(
    cleaned_folder / "dataset_overview.csv",
    index=False
)

features_missing.to_csv(
    cleaned_folder / "missing_values_summary.csv",
    index=False
)

print("Cleaned file saved:", cleaned_file)
print("Final rows:", analysis_data.shape[0])
print("Final columns:", analysis_data.shape[1])

In [ ]:
print("File exists:", cleaned_file.exists())

file_size = cleaned_file.stat().st_size / (1024 * 1024)

print(
    "File size:",
    round(file_size, 2),
    "MB"
)

## 3. Exploratory Data Analysis

Exploratory Data Analysis was used to understand the main characteristics of the historical sales data. Each row represents the weekly sales of one department within one store.

### 3.1 Overall Sales Summary

The following summary presents the number of stores, departments and weeks together with the total, average, median, minimum and maximum sales values.

In [ ]:
total_sales = analysis_data["Weekly_Sales"].sum()
average_sales = analysis_data["Weekly_Sales"].mean()
median_sales = analysis_data["Weekly_Sales"].median()
minimum_sales = analysis_data["Weekly_Sales"].min()
maximum_sales = analysis_data["Weekly_Sales"].max()

store_count = analysis_data["Store"].nunique()
department_count = analysis_data["Dept"].nunique()
week_count = analysis_data["Date"].nunique()

store_department_combinations = (
    analysis_data[["Store", "Dept"]]
    .drop_duplicates()
    .shape[0]
)

overall_summary = pd.DataFrame({
    "Measure": [
        "Sales records",
        "Stores",
        "Departments",
        "Unique weeks",
        "Store-department combinations",
        "Total sales",
        "Average sales per record",
        "Median sales per record",
        "Minimum sales",
        "Maximum sales"
    ],
    "Result": [
        f"{len(analysis_data):,}",
        f"{store_count:,}",
        f"{department_count:,}",
        f"{week_count:,}",
        f"{store_department_combinations:,}",
        f"${total_sales:,.2f}",
        f"${average_sales:,.2f}",
        f"${median_sales:,.2f}",
        f"${minimum_sales:,.2f}",
        f"${maximum_sales:,.2f}"
    ]
})

display(overall_summary)

In [ ]:
sales_statistics = (
    analysis_data["Weekly_Sales"]
    .describe()
    .to_frame(name="Weekly_Sales")
)

sales_statistics["Weekly_Sales"] = (
    sales_statistics["Weekly_Sales"]
    .map(lambda value: f"{value:,.2f}")
)

display(sales_statistics)

### 3.2 Historical Weekly Sales Trend

Weekly sales were added across all stores and departments to examine changes from February 2010 to October 2012. A four-week moving average was included to make the overall trend easier to understand. Holiday weeks were highlighted separately.

In [ ]:
weekly_sales = (
    analysis_data
    .groupby("Date", as_index=False)
    .agg(
        Total_Sales=("Weekly_Sales", "sum"),
        IsHoliday=("IsHoliday", "max")
    )
    .sort_values("Date")
)

weekly_sales["Four_Week_Average"] = (
    weekly_sales["Total_Sales"]
    .rolling(window=4, min_periods=1)
    .mean()
)

print("Number of weeks:", len(weekly_sales))
display(weekly_sales.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    weekly_sales["Date"],
    weekly_sales["Total_Sales"] / 1_000_000,
    color="#9BBAD3",
    linewidth=1.5,
    label="Weekly sales"
)

ax.plot(
    weekly_sales["Date"],
    weekly_sales["Four_Week_Average"] / 1_000_000,
    color="#0B4F6C",
    linewidth=2.5,
    label="Four-week moving average"
)

holiday_points = weekly_sales[
    weekly_sales["IsHoliday"] == True
]

ax.scatter(
    holiday_points["Date"],
    holiday_points["Total_Sales"] / 1_000_000,
    color="#D1495B",
    s=45,
    label="Holiday week",
    zorder=3
)

ax.set_title(
    "Historical Weekly Sales Trend",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Date")
ax.set_ylabel("Total Sales ($ Millions)")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()

weekly_chart_file = (
    charts_folder / "Figure_1_Weekly_Sales_Trend.png"
)

plt.savefig(
    weekly_chart_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Chart saved:", weekly_chart_file)

### 3.3 Sales Peaks and Dips

The highest and lowest sales weeks were identified to understand seasonal demand changes. Holiday names were also included to determine whether major sales peaks occurred during special holiday periods.

In [ ]:
weekly_sales = weekly_sales.drop(
    columns=["Holiday_Name"],
    errors="ignore"
)

holiday_lookup = (
    analysis_data[
        ["Date", "Holiday_Name"]
    ]
    .drop_duplicates()
)

weekly_sales = weekly_sales.merge(
    holiday_lookup,
    on="Date",
    how="left",
    validate="one_to_one"
)

display(weekly_sales.head())

In [ ]:
highest_sales_weeks = (
    weekly_sales
    .nlargest(10, "Total_Sales")
    [["Date", "Total_Sales", "IsHoliday", "Holiday_Name"]]
    .copy()
)

highest_sales_display = highest_sales_weeks.copy()

highest_sales_display["Total_Sales"] = (
    highest_sales_display["Total_Sales"]
    .map(lambda value: f"${value:,.2f}")
)

display(highest_sales_display)

In [ ]:
lowest_sales_weeks = (
    weekly_sales
    .nsmallest(10, "Total_Sales")
    [["Date", "Total_Sales", "IsHoliday", "Holiday_Name"]]
    .copy()
)

lowest_sales_display = lowest_sales_weeks.copy()

lowest_sales_display["Total_Sales"] = (
    lowest_sales_display["Total_Sales"]
    .map(lambda value: f"${value:,.2f}")
)

display(lowest_sales_display)

In [ ]:
highest_sales_weeks.to_csv(
    cleaned_folder / "highest_sales_weeks.csv",
    index=False
)

lowest_sales_weeks.to_csv(
    cleaned_folder / "lowest_sales_weeks.csv",
    index=False
)

print("Peak and dip tables saved.")

#### Interpretation of Sales Peaks and Dips

The highest sales occurred on 24 December 2010, reaching approximately $80.93 million. The second-highest result occurred on 23 December 2011, with approximately $77.00 million. Both weeks were immediately before Christmas, although they were labelled as non-holiday weeks in the dataset. Thanksgiving weeks in 2010 and 2011 also produced large sales peaks.

Most of the lowest sales weeks occurred during January, after the Christmas shopping period. For example, the lowest weekly sales were recorded on 28 January 2011 at approximately $39.60 million. These results show that sales increase before major celebrations and decline after the holiday season. They also show that the IsHoliday variable cannot explain every seasonal change because some important pre-holiday shopping weeks are recorded as non-holidays.

### 3.4 Monthly Seasonal Variation

Average weekly sales were calculated for each month to make a fair comparison between months containing different numbers of sales weeks. This analysis helps identify repeating seasonal patterns.

In [ ]:
weekly_sales["Month_Number"] = weekly_sales["Date"].dt.month
weekly_sales["Month_Name"] = weekly_sales["Date"].dt.month_name()

monthly_seasonality = (
    weekly_sales
    .groupby(
        ["Month_Number", "Month_Name"],
        as_index=False
    )
    .agg(
        Average_Weekly_Sales=("Total_Sales", "mean"),
        Minimum_Weekly_Sales=("Total_Sales", "min"),
        Maximum_Weekly_Sales=("Total_Sales", "max"),
        Weeks_Observed=("Date", "count")
    )
    .sort_values("Month_Number")
)

monthly_display = monthly_seasonality.copy()

for column in [
    "Average_Weekly_Sales",
    "Minimum_Weekly_Sales",
    "Maximum_Weekly_Sales"
]:
    monthly_display[column] = monthly_display[column].map(
        lambda value: f"${value:,.2f}"
    )

display(monthly_display)

In [ ]:
highest_month = monthly_seasonality.loc[
    monthly_seasonality["Average_Weekly_Sales"].idxmax()
]

lowest_month = monthly_seasonality.loc[
    monthly_seasonality["Average_Weekly_Sales"].idxmin()
]

print(
    "Highest average month:",
    highest_month["Month_Name"],
    f"${highest_month['Average_Weekly_Sales']:,.2f}"
)

print(
    "Lowest average month:",
    lowest_month["Month_Name"],
    f"${lowest_month['Average_Weekly_Sales']:,.2f}"
)

In [ ]:
monthly_values = (
    monthly_seasonality["Average_Weekly_Sales"] / 1_000_000
)

highest_value = monthly_values.max()
lowest_value = monthly_values.min()

bar_colours = [
    "#D1495B" if value == highest_value
    else "#AAB7C4" if value == lowest_value
    else "#2E74B5"
    for value in monthly_values
]

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(
    monthly_seasonality["Month_Name"],
    monthly_values,
    color=bar_colours
)

ax.bar_label(
    bars,
    labels=[
        f"{value:.1f}"
        for value in monthly_values
    ],
    padding=3,
    fontsize=8
)

ax.set_title(
    "Average Weekly Sales by Month",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Month")
ax.set_ylabel("Average Weekly Sales ($ Millions)")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.xticks(rotation=45)
plt.tight_layout()

monthly_chart_file = (
    charts_folder / "Figure_2_Monthly_Seasonality.png"
)

plt.savefig(
    monthly_chart_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Chart saved:", monthly_chart_file)

In [ ]:
monthly_seasonality.to_csv(
    cleaned_folder / "monthly_seasonality.csv",
    index=False
)

print("Monthly results saved.")

#### Monthly Seasonality Findings

December had the highest average weekly sales at approximately $57.7 million, while January had the lowest at approximately $41.6 million. November also recorded high average sales of around $51.6 million. This shows that customer spending increases before Christmas and then falls after the holiday season. The results suggest that the company should prepare more stock and staff for November and December. However, it should plan more carefully in January because demand is lower.

### 3.5 Holiday and Non-Holiday Sales Comparison

Average total sales per week were compared between holiday and non-holiday periods. Each date was counted only once so that the comparison was fair.

In [ ]:
holiday_comparison = (
    weekly_sales
    .groupby("IsHoliday", as_index=False)
    .agg(
        Average_Weekly_Sales=("Total_Sales", "mean"),
        Minimum_Weekly_Sales=("Total_Sales", "min"),
        Maximum_Weekly_Sales=("Total_Sales", "max"),
        Weeks_Observed=("Date", "nunique")
    )
)

holiday_comparison["Week_Type"] = holiday_comparison["IsHoliday"].map(
    {False: "Non-holiday", True: "Holiday"}
)

holiday_display = holiday_comparison.copy()

for column in [
    "Average_Weekly_Sales",
    "Minimum_Weekly_Sales",
    "Maximum_Weekly_Sales"
]:
    holiday_display[column] = holiday_display[column].map(
        lambda value: f"${value:,.2f}"
    )

display(
    holiday_display[
        [
            "Week_Type",
            "Average_Weekly_Sales",
            "Minimum_Weekly_Sales",
            "Maximum_Weekly_Sales",
            "Weeks_Observed"
        ]
    ]
)

In [ ]:
holiday_plot = holiday_comparison.sort_values("IsHoliday").copy()
holiday_plot["Sales_Millions"] = (
    holiday_plot["Average_Weekly_Sales"] / 1_000_000
)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    holiday_plot["Week_Type"],
    holiday_plot["Sales_Millions"],
    color=["#AAB7C4", "#D1495B"]
)

ax.bar_label(
    bars,
    labels=[
        f"${value:.1f}M"
        for value in holiday_plot["Sales_Millions"]
    ],
    padding=3,
    fontweight="bold"
)

ax.set_title(
    "Average Weekly Sales: Holiday vs Non-Holiday",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Week Type")
ax.set_ylabel("Average Weekly Sales ($ Millions)")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()

holiday_chart_file = (
    charts_folder / "Figure_3_Holiday_vs_Non_Holiday.png"
)

plt.savefig(
    holiday_chart_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

holiday_comparison.to_csv(
    cleaned_folder / "holiday_sales_comparison.csv",
    index=False
)

print("Chart and results saved.")

#### Holiday Sales Findings

Average total sales were approximately $50.5 million during holiday weeks and $46.9 million during non-holiday weeks. This means holiday weeks produced about 7.7% higher sales. The result shows that holidays influence customer spending. The company should prepare enough stock and staff before important holidays. However, each holiday may affect sales differently, so the four major holidays will also be compared separately.

### 3.6 Sales Comparison Across Major Holidays

Historical average weekly sales were compared across the Super Bowl, Labour Day, Thanksgiving and Christmas. This helps identify which holiday periods had the strongest and weakest sales performance.

In [ ]:
major_holidays = [
    "Super Bowl",
    "Labour Day",
    "Thanksgiving",
    "Christmas"
]

major_holiday_data = weekly_sales[
    weekly_sales["Holiday_Name"].isin(major_holidays)
].copy()

major_holiday_summary = (
    major_holiday_data
    .groupby("Holiday_Name", as_index=False)
    .agg(
        Average_Weekly_Sales=("Total_Sales", "mean"),
        Minimum_Weekly_Sales=("Total_Sales", "min"),
        Maximum_Weekly_Sales=("Total_Sales", "max"),
        Weeks_Observed=("Date", "nunique")
    )
)

major_holiday_summary["Holiday_Name"] = pd.Categorical(
    major_holiday_summary["Holiday_Name"],
    categories=major_holidays,
    ordered=True
)

major_holiday_summary = (
    major_holiday_summary
    .sort_values("Holiday_Name")
    .reset_index(drop=True)
)

holiday_summary_display = major_holiday_summary.copy()

for column in [
    "Average_Weekly_Sales",
    "Minimum_Weekly_Sales",
    "Maximum_Weekly_Sales"
]:
    holiday_summary_display[column] = (
        holiday_summary_display[column]
        .map(lambda value: f"${value:,.2f}")
    )

display(holiday_summary_display)

In [ ]:
highest_holiday = major_holiday_summary.loc[
    major_holiday_summary["Average_Weekly_Sales"].idxmax()
]

lowest_holiday = major_holiday_summary.loc[
    major_holiday_summary["Average_Weekly_Sales"].idxmin()
]

print(
    "Highest average holiday:",
    highest_holiday["Holiday_Name"],
    f"${highest_holiday['Average_Weekly_Sales']:,.2f}"
)

print(
    "Lowest average holiday:",
    lowest_holiday["Holiday_Name"],
    f"${lowest_holiday['Average_Weekly_Sales']:,.2f}"
)

In [ ]:
holiday_values = (
    major_holiday_summary["Average_Weekly_Sales"] / 1_000_000
)

highest_value = holiday_values.max()
lowest_value = holiday_values.min()

bar_colours = [
    "#D1495B" if value == highest_value
    else "#AAB7C4" if value == lowest_value
    else "#2E74B5"
    for value in holiday_values
]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    major_holiday_summary["Holiday_Name"].astype(str),
    holiday_values,
    color=bar_colours
)

ax.bar_label(
    bars,
    labels=[f"${value:.1f}M" for value in holiday_values],
    padding=3,
    fontweight="bold"
)

ax.set_title(
    "Average Weekly Sales by Major Holiday",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Holiday")
ax.set_ylabel("Average Weekly Sales ($ Millions)")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()

major_holiday_chart_file = (
    charts_folder / "Figure_4_Major_Holiday_Sales.png"
)

plt.savefig(
    major_holiday_chart_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

major_holiday_summary.to_csv(
    cleaned_folder / "major_holiday_sales_summary.csv",
    index=False
)

print("Holiday chart and results saved.")

#### Major Holiday Findings

Thanksgiving recorded the highest average weekly sales at approximately $66.2 million. It was followed by the Super Bowl at $48.6 million and Labour Day at $46.9 million. The Christmas-labelled weeks had the lowest average at $43.2 million. However, this does not mean the whole Christmas season performed poorly. The earlier analysis showed that sales peaked during the weeks before Christmas. The holiday results should also be interpreted carefully because only two or three weeks were available for each holiday.

## 4. Store-Wise Analysis

### 4.1 Highest and Lowest Selling Stores

Total sales were calculated for each of the 45 stores across the complete historical period. The stores were then ranked to identify the highest- and lowest-selling locations.

In [ ]:
store_sales = (
    analysis_data
    .groupby(["Store", "Type", "Size"], as_index=False)
    .agg(
        Total_Sales=("Weekly_Sales", "sum"),
        Average_Sales_Per_Record=("Weekly_Sales", "mean"),
        Departments=("Dept", "nunique"),
        Weeks_Observed=("Date", "nunique")
    )
    .sort_values("Total_Sales", ascending=False)
    .reset_index(drop=True)
)

store_sales["Sales_Rank"] = (
    store_sales["Total_Sales"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

top_10_stores = store_sales.head(10).copy()

bottom_10_stores = (
    store_sales
    .sort_values("Total_Sales", ascending=True)
    .head(10)
    .copy()
)

def format_store_results(table):
    formatted_table = table.copy()

    formatted_table["Total_Sales"] = (
        formatted_table["Total_Sales"]
        .map(lambda value: f"${value:,.2f}")
    )

    formatted_table["Average_Sales_Per_Record"] = (
        formatted_table["Average_Sales_Per_Record"]
        .map(lambda value: f"${value:,.2f}")
    )

    formatted_table["Size"] = (
        formatted_table["Size"]
        .map(lambda value: f"{value:,}")
    )

    return formatted_table

print("Top 10 highest-selling stores:")
display(format_store_results(top_10_stores))

print("Top 10 lowest-selling stores:")
display(format_store_results(bottom_10_stores))

In [ ]:
highest_store = store_sales.iloc[0]

lowest_store = store_sales.sort_values(
    "Total_Sales",
    ascending=True
).iloc[0]

print(
    f"Highest-selling store: Store {int(highest_store['Store'])} "
    f"with ${highest_store['Total_Sales']:,.2f}"
)

print(
    f"Lowest-selling store: Store {int(lowest_store['Store'])} "
    f"with ${lowest_store['Total_Sales']:,.2f}"
)

store_sales.to_csv(
    cleaned_folder / "store_sales_ranking.csv",
    index=False
)

print("Store ranking saved.")

In [ ]:
top_5_stores = (
    store_sales
    .nlargest(5, "Total_Sales")
    .assign(Store_Group="Top 5")
)

bottom_5_stores = (
    store_sales
    .nsmallest(5, "Total_Sales")
    .assign(Store_Group="Bottom 5")
)

store_chart_data = (
    pd.concat([top_5_stores, bottom_5_stores])
    .drop_duplicates(subset="Store")
    .sort_values("Total_Sales")
    .copy()
)

store_chart_data["Store_Label"] = (
    "Store " + store_chart_data["Store"].astype(int).astype(str)
)

store_chart_data["Sales_Millions"] = (
    store_chart_data["Total_Sales"] / 1_000_000
)

bar_colours = [
    "#D1495B" if store == int(highest_store["Store"])
    else "#AAB7C4" if group == "Bottom 5"
    else "#2E74B5"
    for store, group in zip(
        store_chart_data["Store"],
        store_chart_data["Store_Group"]
    )
]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    store_chart_data["Store_Label"],
    store_chart_data["Sales_Millions"],
    color=bar_colours
)

ax.bar_label(
    bars,
    labels=[
        f"${value:.1f}M"
        for value in store_chart_data["Sales_Millions"]
    ],
    padding=4,
    fontsize=9
)

ax.set_title(
    "Top and Bottom Five Stores by Total Sales",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Total Sales ($ Millions)")
ax.set_ylabel("Store")
ax.set_xlim(0, store_chart_data["Sales_Millions"].max() * 1.15)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()

store_chart_file = (
    charts_folder / "Figure_5_Store_Sales_Ranking.png"
)

plt.savefig(
    store_chart_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Store ranking chart saved:", store_chart_file)

#### Store-Wise Findings

Store 20 recorded the highest total sales at approximately $301.4 million. Stores 4, 14, 13 and 2 were also among the five highest-selling stores. Store 33 recorded the lowest total sales at approximately $37.2 million. Store 20 generated about eight times more sales than Store 33. However, total sales alone do not explain why these differences occurred. Store size, store type and the number of active departments may also influence each store’s performance.

### 4.2 Sales Performance by Store Type

The stores were grouped into Types A, B and C. Average sales per store were used for a fair comparison because the three groups contain different numbers of stores.

In [ ]:
store_type_summary = (
    store_sales
    .groupby("Type", as_index=False)
    .agg(
        Number_of_Stores=("Store", "nunique"),
        Average_Store_Size=("Size", "mean"),
        Total_Sales=("Total_Sales", "sum"),
        Average_Sales_Per_Store=("Total_Sales", "mean")
    )
    .sort_values("Average_Sales_Per_Store", ascending=False)
)

store_type_display = store_type_summary.copy()

store_type_display["Average_Store_Size"] = (
    store_type_display["Average_Store_Size"]
    .map(lambda value: f"{value:,.0f}")
)

store_type_display["Total_Sales"] = (
    store_type_display["Total_Sales"]
    .map(lambda value: f"${value:,.2f}")
)

store_type_display["Average_Sales_Per_Store"] = (
    store_type_display["Average_Sales_Per_Store"]
    .map(lambda value: f"${value:,.2f}")
)

display(store_type_display)

store_type_summary.to_csv(
    cleaned_folder / "store_type_sales_summary.csv",
    index=False
)

print("Store type summary saved.")

In [ ]:
type_values = (
    store_type_summary["Average_Sales_Per_Store"] / 1_000_000
)

highest_type_value = type_values.max()
lowest_type_value = type_values.min()

type_colours = [
    "#D1495B" if value == highest_type_value
    else "#AAB7C4" if value == lowest_type_value
    else "#2E74B5"
    for value in type_values
]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    store_type_summary["Type"],
    type_values,
    color=type_colours
)

ax.bar_label(
    bars,
    labels=[f"${value:.1f}M" for value in type_values],
    padding=3,
    fontweight="bold"
)

ax.set_title(
    "Average Total Sales per Store by Store Type",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Store Type")
ax.set_ylabel("Average Sales per Store ($ Millions)")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()

store_type_chart_file = (
    charts_folder / "Figure_6_Store_Type_Sales.png"
)

plt.savefig(
    store_type_chart_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Store type chart saved:", store_type_chart_file)

#### Store-Type Findings

Type A stores recorded the highest average total sales at approximately $196.9 million per store. Type B stores averaged $117.7 million, while Type C stores averaged $67.6 million. A Type A store therefore generated almost three times the average sales of a Type C store. However, this does not prove that store type alone caused the difference. Store size, location and the number of departments may also affect sales performance.

### 4.3 Relationship Between Store Size and Sales

The relationship between store size and total sales was examined using Pearson correlation and a scatter plot. A positive correlation would mean that larger stores generally record higher sales.

In [ ]:
size_sales_correlation = store_sales["Size"].corr(
    store_sales["Total_Sales"]
)

print(
    f"Correlation between store size and total sales: "
    f"{size_sales_correlation:.3f}"
)

In [ ]:
store_size_plot = store_sales.copy()

store_size_plot["Sales_Millions"] = (
    store_size_plot["Total_Sales"] / 1_000_000
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=store_size_plot,
    x="Size",
    y="Sales_Millions",
    hue="Type",
    palette={
        "A": "#D1495B",
        "B": "#2E74B5",
        "C": "#AAB7C4"
    },
    s=90,
    ax=ax
)

sns.regplot(
    data=store_size_plot,
    x="Size",
    y="Sales_Millions",
    scatter=False,
    color="#333333",
    line_kws={"linestyle": "--", "linewidth": 2},
    ax=ax
)

ax.set_title(
    "Relationship Between Store Size and Total Sales",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Store Size")
ax.set_ylabel("Total Sales ($ Millions)")
ax.grid(alpha=0.25)
ax.set_axisbelow(True)
ax.legend(title="Store Type")

plt.tight_layout()

store_size_chart_file = (
    charts_folder / "Figure_7_Store_Size_vs_Sales.png"
)

plt.savefig(
    store_size_chart_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Store size chart saved:", store_size_chart_file)

#### Store Size Findings

The scatter plot shows a clear positive relationship between store size and total sales. Larger stores generally recorded higher sales, while smaller stores usually recorded lower sales. Most Type A stores were located among the larger and higher-selling stores, while Type C stores were generally smaller. This helps explain some of the differences in the store rankings. However, correlation does not prove that store size alone caused higher sales because location, departments and customer demand may also have influenced performance.

## 5. AI-Assisted EDA Comparison

ChatGPT was given seven charts from the completed EDA on 3 September 2026. It was asked to identify sales trends, seasonal changes, peaks, dips, holiday effects and store differences. The response was checked against the Python results.

| Area | Own EDA Findings | ChatGPT Findings | Comparison |
|---|---|---|---|
| General trend | Weekly sales were mostly stable, with strong year-end peaks. | Sales were generally around $40–$50 million with year-end peaks. | Both results agreed. |
| Seasonality | December was highest and January was lowest. | December was highest and January was lowest. | Both identified the same seasonal pattern. |
| Holidays | Holiday weeks were 7.7% higher, and Thanksgiving was strongest. | Holiday weeks were higher, and Thanksgiving was strongest. | Both results agreed. |
| Stores | Store 20 was highest, Store 33 was lowest, and larger stores generally sold more. | ChatGPT identified the same store results. | The AI confirmed the Python findings. |
| Limitations | Holiday samples were small, and Christmas shopping peaked before the labelled Christmas week. | The AI mentioned the short period and that EDA cannot explain causes or predict future sales. | The limitations were different but both were useful. |

Overall, ChatGPT provided a clear summary that supported the main Python findings. However, the analysis’s analysis gave more detail about the four holidays, store types and the unusual Christmas-week result. Therefore, the AI response was used as a comparison and was not accepted without checking the original tables and charts.